消息裁剪强调“在模型调用前裁剪消息列表，控制模型可以看到的上下文范围”，而消息删除强调模型
调用完成后将某些消息从消息列表中移除，永久更改状态。
适合明确要遗忘、清理、重置某些历史。

In [1]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:

    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，老王。从现在起我叫小王。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，天气不错的时候心情也会跟着轻松一些。你是准备出去走走，还是想聊点别的？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是小王。  
你是老王。
